# **Notebook 3: Baseline Model Evaluation**
## Capstone: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
from pathlib import Path
import json, re
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
test_query = 'My package still has not arrived and the tracking page has not moved. When will I get it?'
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    quant_config = BitsAndBytesConfig(load_in_4bit=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map='auto')
    model.eval(); model_available = True
    print('Loaded base model in 4-bit:', MODEL_ID)
except Exception as exc:
    print('Model stack unavailable; using deterministic baseline stub:', repr(exc))
    tokenizer = None; model = None; model_available = False


Model stack unavailable; using deterministic baseline stub: ModuleNotFoundError("No module named 'torch'")


In [2]:
def generate_baseline_response(query, max_new_tokens=160):
    if model_available:
        messages=[{'role':'user','content':query}]
        prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs=tokenizer(prompt, return_tensors='pt').to(model.device)
        outputs=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None)
        return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    return 'Your order is probably delayed. Most packages arrive within about 5 to 10 business days, but you should check the tracking portal or contact the shipping department for a precise date.'
baseline_output = generate_baseline_response(test_query)
print('Query:', test_query)
print('Baseline output:\n', baseline_output)


Query: My package still has not arrived and the tracking page has not moved. When will I get it?
Baseline output:
 Your order is probably delayed. Most packages arrive within about 5 to 10 business days, but you should check the tracking portal or contact the shipping department for a precise date.


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
ground_truth_rule = 'Domestic orders deliver within 3-7 business days; international orders within 10-21 business days.'
unsupported_flags=[]
if not re.search(r'3\s*[\-?]\s*7|10\s*[\-?]\s*21|24|48|5 business days', baseline_output, re.I): unsupported_flags.append('does not cite the SOP delivery windows')
if re.search(r'5\s*(to|-)\s*10|shipping department|precise date|guaranteed', baseline_output, re.I): unsupported_flags.append('contains a likely invented timeline or unsupported action')
baseline_eval={'ground_truth_rule':ground_truth_rule,'baseline_output':baseline_output,'hallucinations_or_gaps':unsupported_flags,'assessment':'Baseline is unsupported or incomplete' if unsupported_flags else 'Baseline aligns with the SOP rule'}
print(json.dumps(baseline_eval, indent=2))


{
  "ground_truth_rule": "Domestic orders deliver within 3-7 business days; international orders within 10-21 business days.",
  "baseline_output": "Your order is probably delayed. Most packages arrive within about 5 to 10 business days, but you should check the tracking portal or contact the shipping department for a precise date.",
  "hallucinations_or_gaps": [
    "does not cite the SOP delivery windows",
    "contains a likely invented timeline or unsupported action"
  ],
  "assessment": "Baseline is unsupported or incomplete"
}


---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
outputs_path=Path('outputs.json')
outputs=json.loads(outputs_path.read_text(encoding='utf-8')) if outputs_path.exists() else {}
outputs.update({'test_query':test_query,'baseline_output':baseline_output,'baseline_eval':baseline_eval})
outputs_path.write_text(json.dumps(outputs, indent=2), encoding='utf-8')
print(f'Saved baseline artifact to {outputs_path.resolve()}')


Saved baseline artifact to C:\Users\sysadmin\Downloads\Capestone project\Starter Files\outputs.json


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**